# Building the feeder step by step

This notebook reads `data/sample/kt1_sample.csv` (built by `sampler_dev.ipynb`) and drips it into `data/stream_input/` as small files over time — simulating events arriving live for `streaming_job_dev.ipynb` to consume.

Run `streaming_job_dev.ipynb` first (or side by side) so a consumer is ready when files start landing.

In [7]:
import sys
sys.path.insert(0, "../src")

import os
import shutil
import time
from pathlib import Path

import pandas as pd
from config import (
    SAMPLE_FILE,
    STREAM_INPUT_DIR,
    CHECKPOINT_DIR,
    KT1_STREAM_COLUMNS,
    DEFAULT_BATCH_ROWS,
    DEFAULT_FEED_INTERVAL_SEC,
)

pd.set_option("display.max_columns", None)

In [8]:
DTYPES = {
    "timestamp": "int64",
    "solving_id": "int64",
    "question_id": "string",
    "user_answer": "string",
    "elapsed_time": "int64",
    "user_id": "string",
}

sample = pd.read_csv(SAMPLE_FILE, dtype=DTYPES)
print(sample.shape)
sample.head()

(43574, 6)


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
0,1498906740412,1,q129,b,27000,u22094
1,1498906764080,2,q8058,b,22000,u22094
2,1498906786376,3,q8120,a,20000,u22094
3,1498906818824,4,q157,d,30000,u22094
4,1498906848124,5,q52,b,27000,u22094


## Step 1 — split the sample into ordered row-chunks

We need to slice `sample` into consecutive pieces of `batch_rows` rows each — **preserving the timestamp order** the sampler already sorted (don't shuffle). Each chunk becomes one file that lands in `data/stream_input/` later.

Fill in the two blanks: `range(start, stop, step)` needs a step size, and each slice needs an end position. Both should be `batch_rows`.

In [ ]:
def chunk_dataframe(df: pd.DataFrame, batch_rows: int):
    for start in range(0, len(df), batch_rows):  # TODO: fill in
        yield df.iloc[start : start + batch_rows]  # TODO: fill in


chunks = list(chunk_dataframe(sample, DEFAULT_BATCH_ROWS))
print(len(chunks), "chunks")
print("first chunk:", chunks[0].shape, "last chunk:", chunks[-1].shape)

218 chunks
first chunk: (200, 6) last chunk: (174, 6)


## Step 2 — write one chunk to disk, atomically

Spark's streaming file source polls `data/stream_input/` and could read a file while it's still being written, if we write directly to the final filename. The fix: write to a hidden temp name first, then atomically rename it to the real name. Spark's directory listing never sees a half-written file this way (and files starting with `.` are ignored by Spark's file source anyway, as a backup safety net).

Fill in the three blanks:
1. Which path should `chunk.to_csv(...)` write to — the temp path or the final path?
2. `os.replace(src, dst)` renames `src` to `dst`. Which is which here?"

In [ ]:
def write_batch(chunk: pd.DataFrame, seq: int, out_dir: Path) -> Path:
    final_path = out_dir / f"part-{seq:06d}.csv"
    tmp_path = out_dir / f".tmp-part-{seq:06d}.csv"

    chunk.to_csv(tmp_path, index=False)  # TODO: write to which path first?
    os.replace(tmp_path, final_path)            # TODO: rename which path to which?

    return final_path


STREAM_INPUT_DIR.mkdir(parents=True, exist_ok=True)
test_path = write_batch(chunks[0], 0, STREAM_INPUT_DIR)
print(test_path, test_path.exists())
pd.read_csv(test_path, dtype=DTYPES).head()

/home/verseua/projects/PySpark-Data-Processing/data/stream_input/part-000000.csv True


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
0,1498906740412,1,q129,b,27000,u22094
1,1498906764080,2,q8058,b,22000,u22094
2,1498906786376,3,q8120,a,20000,u22094
3,1498906818824,4,q157,d,30000,u22094
4,1498906848124,5,q52,b,27000,u22094


## Step 3 — reset between demo runs

Between runs we want a clean slate: delete leftover `.csv` files in `data/stream_input/`, **and** delete Spark's checkpoint directories. Reason: a checkpoint remembers which files were already processed — if you wipe the input files but leave an old checkpoint around, a fresh run can get confusing offset bookkeeping. Clearing both together keeps every run reproducible.

Fill in the one blank: which `shutil` function deletes an entire directory tree (not just a single file)?

In [ ]:
def reset_dir(out_dir: Path, checkpoint_dir: Path) -> None:
    csv_files = list(out_dir.glob("*.csv"))
    print(f"Deleting {len(csv_files)} existing files in {out_dir}")
    for f in csv_files:
        f.unlink()

    if checkpoint_dir.exists():
        print(f"Deleting checkpoint dir {checkpoint_dir}")
        shutil.rmtree(checkpoint_dir)  # TODO: which shutil function?
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

**Run this cell now, once, before starting the queries in `streaming_job_dev.ipynb`.** Do not re-run it later while the queries are running — it deletes the checkpoint directories they depend on and will crash them with a `FileNotFoundError` (this is exactly what just happened when the old feed-loop cell called `reset_dir` after the queries had already started).

In [ ]:
reset_dir(STREAM_INPUT_DIR, CHECKPOINT_DIR)

## Step 4 — run the feed loop

Writes one chunk at a time with a pause in between, so files trickle into `data/stream_input/` like a live feed. **Reset already happened in the cell above** — this cell no longer touches checkpoints, so it's safe to re-run this cell alone if you want to feed again without restarting the streaming job (though the consumer will have already processed the earlier files, so you'd be feeding the same data twice — fine for testing the feeder alone, not for a clean demo run).

Make sure `streaming_job_dev.ipynb`'s queries are already started before running this cell.

Fill in the four blanks: the three arguments `write_batch` needs (which chunk, which sequence number, which output directory), and which config constant controls how long to pause between batches.

This cell blocks while it runs — use the notebook's interrupt/stop button to stop early if you want; already-written files stay in place.

In [ ]:
all_chunks = list(chunk_dataframe(sample, DEFAULT_BATCH_ROWS))
total = len(all_chunks)

for seq, chunk in enumerate(all_chunks):
    path = write_batch(chunk, seq, STREAM_INPUT_DIR)
    print(f"wrote {path.name} ({len(chunk)} rows) — {seq + 1}/{total}")
    time.sleep(DEFAULT_FEED_INTERVAL_SEC)